## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Chip bag weights: each line against the 1 g rule

**Hypothesis Testing for the Variance** · a worked example from the StatNex teaching guide

**The story.** A snack factory fills 150 g bags of chips on two packing lines. Its quality rule says the standard deviation of bag weight must not exceed 1 g, otherwise too many bags are underfilled or the company gives away free chips. Fourteen bags were weighed from each line.

**The question.** Does either packing line break the 1 g consistency rule?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_variance_test.csv` (14 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Line A,Line B
1.0,150.3,152.2
2.0,151.4,149.5
3.0,151.0,151.3
4.0,149.2,151.4
5.0,151.2,151.2
6.0,150.9,150.2
7.0,150.7,155.4
8.0,151.4,150.2
9.0,151.2,153.6
10.0,150.4,153.7
11.0,150.6,148.2
12.0,151.3,151.6
13.0,150.8,153.3
14.0,151.9,149.6
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Line A", "Line B"]
# The population standard deviation under the null hypothesis.
sigma0 = 1.0

## Chi-square test for the variance, one column at a time

In [ ]:
# chi-square = (n - 1) s² / σ0²: if the true SD were σ0, it would land near n - 1, its
# degrees of freedom. Far above means more spread than σ0, far below means less.
# SciPy has no one-line test for this, so the chi-square distribution is used directly.
# The test assumes normal data and, unlike the t test, does not become safe as n
# grows: heavy tails make it reject far more often than it should.
results = {}
for name in variables:
    # Text and blank cells count as missing and are skipped.
    x = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
    n, sd = len(x), x.std(ddof=1)
    df = n - 1
    variance = sd**2
    ratio = variance / sigma0**2
    chi2 = df * ratio
    # The one-sided P value is the smaller tail -- the side the observed variance lies
    # on -- and the two-sided P value is twice that, capped at 1 (Stata's sdtest does
    # the same). Use the one-sided P only if its direction was fixed in advance.
    p_one = min(stats.chi2.cdf(chi2, df), stats.chi2.sf(chi2, df))
    p_two = min(1.0, 2 * p_one)
    results[name] = {"n": n, "SD": sd, "variance": variance, "sigma0^2": sigma0**2,
                     "ratio": ratio, "chi-square": chi2, "df": df,
                     "P two-sided": p_two, "P one-sided": p_one}

table = pd.DataFrame(results)
print(table)
print()
# Stars go on the two-sided P only: the one-sided P takes the direction the data lie
# in, so starring it would halve the two-sided P after looking.
print(table.loc[["P two-sided"]].map(stars))

## What the result says

The standard deviation, the typical distance of a bag from its line's average, is 0.65 g on Line A and 1.97 g on Line B. The test compares each line's variance (the standard deviation squared) with the variance the rule allows, which is 1.

Line B's variance is 3.86 times the limit. Its chi-square statistic is 50.2 and the two-sided P value is 0.000005. A P value is the chance of a result at least this extreme if the line's true standard deviation were exactly 1 g. Line B breaks the rule.

Line A's ratio is 0.42, with chi-square 5.48. Its one-sided P value is 0.037 and its two-sided P value is 0.074. So Line A stays within the rule, but 14 bags give only moderate evidence of that.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.